# 09 — Synthèse multi-sources

Tableau de bord consolidé croisant CNAV, INSEE (pyramide + espérance de vie) et DREES (tous régimes).

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import sqlalchemy as sa
from src.db import engine

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10
conn = engine().connect()

## 1. KPIs 2023 — tableau de synthèse

In [ ]:
kpi = pd.read_sql("""
    SELECT
        d.annee,
        d.retraites_total_cnav,
        d.pension_moy_cnav,
        d.age_moyen_depart,
        d.inflation_pct,
        d.esperance_vie_60_f,
        d.esperance_vie_60_h,
        c.pension_brute_drees_t   AS pension_drees_tous_reg,
        c.pension_brute_drees_f   AS pension_drees_f,
        c.pension_brute_drees_h   AS pension_drees_h,
        c.part_cnav_pct
    FROM rpt.v_DashboardPrincipal d
    LEFT JOIN rpt.v_CompaisonRegimes c ON c.annee = d.annee
    WHERE d.annee = 2023
""", conn)

print("=== KPIs 2023 ===")
for col in kpi.columns:
    v = kpi[col].iloc[0]
    print(f"  {col:40s} : {v}")

## 2. Tableau de bord 2000–2023 — 6 indicateurs

In [ ]:
dash = pd.read_sql("""
    SELECT d.annee,
           d.retraites_total_cnav,
           d.pension_moy_cnav,
           d.age_moyen_depart,
           d.inflation_pct,
           d.esperance_vie_60_f,
           d.esperance_vie_60_h,
           d.reforme_annee,
           c.pension_brute_drees_t,
           c.part_cnav_pct
    FROM rpt.v_DashboardPrincipal d
    LEFT JOIN rpt.v_CompaisonRegimes c ON c.annee = d.annee
    WHERE d.annee BETWEEN 2000 AND 2023
    ORDER BY d.annee
""", conn)

for col in ['pension_moy_cnav', 'pension_brute_drees_t', 'inflation_pct', 'esperance_vie_60_f', 'esperance_vie_60_h', 'part_cnav_pct']:
    dash[col] = pd.to_numeric(dash[col], errors='coerce')

reformes = dash[dash['reforme_annee'].notna()][['annee', 'reforme_annee']]

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

def add_reformes(ax):
    for _, r in reformes.iterrows():
        ax.axvline(r['annee'], color='red', ls='--', lw=0.7, alpha=0.6)
        ax.text(r['annee'] + 0.1, ax.get_ylim()[1] * 0.97,
                r['reforme_annee'], rotation=90, fontsize=6.5, color='red', va='top')

# 1. Effectifs CNAV
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(dash['annee'], dash['retraites_total_cnav'] / 1e6, color='steelblue', lw=2)
ax1.set_title('Effectifs CNAV (millions)'); ax1.grid(alpha=0.3)
add_reformes(ax1)

# 2. Pension CNAV vs DREES
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(dash['annee'], dash['pension_moy_cnav'], label='CNAV', color='navy', lw=2)
ax2.plot(dash['annee'], dash['pension_brute_drees_t'], label='DREES tous régimes', color='coral', lw=2, ls='--')
ax2.set_title('Pension mensuelle brute (€ courants)'); ax2.legend(); ax2.grid(alpha=0.3)
add_reformes(ax2)

# 3. Âge moyen de départ
ax3 = fig.add_subplot(gs[1, 0])
age_dep = pd.read_sql("""
    SELECT annee, genre_code, valeur FROM ext.DREES_PensionsMultiRegimes
    WHERE indicateur = 'AGE_CONJONCTUREL_DEPART' ORDER BY annee, genre_code
""", conn).pivot(index='annee', columns='genre_code', values='valeur').apply(pd.to_numeric, errors='coerce')
for gc, color in [('F', 'salmon'), ('H', 'steelblue'), ('T', 'gray')]:
    if gc in age_dep.columns:
        ax3.plot(age_dep.index, age_dep[gc], color=color, label={'F':'F','H':'H','T':'Ensemble'}[gc])
ax3.set_title('Âge conjoncturel de départ (tous régimes)'); ax3.legend(); ax3.grid(alpha=0.3)
add_reformes(ax3)

# 4. Espérance de vie à 60 ans
ax4 = fig.add_subplot(gs[1, 1])
ev_all = pd.read_sql("""
    SELECT annee, genre_code, esperance_annees FROM ext.INSEE_EsperanceVie
    WHERE age_reference = 60 AND annee BETWEEN 2000 AND 2023 ORDER BY annee, genre_code
""", conn)
for gc, color in [('F', 'salmon'), ('H', 'steelblue')]:
    sub = ev_all[ev_all['genre_code'] == gc]
    ax4.plot(sub['annee'], sub['esperance_annees'], color=color, lw=2, label=gc)
ax4.set_title('Espérance de vie à 60 ans (années)'); ax4.legend(); ax4.grid(alpha=0.3)
add_reformes(ax4)

# 5. Inflation
ax5 = fig.add_subplot(gs[2, 0])
ax5.bar(dash['annee'], dash['inflation_pct'], color='darkorange', alpha=0.7)
ax5.set_title('Inflation (%)'); ax5.grid(axis='y', alpha=0.3)

# 6. Part CNAV dans pension totale
ax6 = fig.add_subplot(gs[2, 1])
ax6.fill_between(dash['annee'], pd.to_numeric(dash['part_cnav_pct'], errors='coerce'),
                 alpha=0.4, color='navy')
ax6.plot(dash['annee'], pd.to_numeric(dash['part_cnav_pct'], errors='coerce'), color='navy', lw=2)
ax6.set_title('Part CNAV / pension totale DREES (%)'); ax6.grid(alpha=0.3)

fig.suptitle('Tableau de bord retraites — Multi-sources (CNAV · INSEE · DREES) — 2000–2023',
             fontsize=13, fontweight='bold')
plt.savefig('reports/09_dashboard_multi_sources.png', bbox_inches='tight')
plt.show()

## 3. Durée de retraite estimée — espérance de vie vs âge de départ

In [ ]:
duree = pd.read_sql("""
    SELECT annee, genre, age_depart, esperance_vie_a_60, duree_retraite_estimee_ans
    FROM rpt.v_DureeRetraiteEstimee
    WHERE annee BETWEEN 1994 AND 2023 AND esperance_vie_a_60 IS NOT NULL
    ORDER BY annee, genre
""", conn)

for col in ['age_depart', 'esperance_vie_a_60', 'duree_retraite_estimee_ans']:
    duree[col] = pd.to_numeric(duree[col], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Durée estimée de retraite
for genre, color in [('Total', 'gray')]:
    sub = duree[duree['genre'] == genre]
    if len(sub):
        axes[0].fill_between(sub['annee'], sub['duree_retraite_estimee_ans'], alpha=0.3, color=color)
        axes[0].plot(sub['annee'], sub['duree_retraite_estimee_ans'], color=color, lw=2, label=genre)

axes[0].set_title('Durée estimée de retraite (ans)', fontweight='bold')
axes[0].set_ylabel('Années'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Croisement espérance de vie à 60 vs âge de départ
ev_full = pd.read_sql("""
    SELECT annee, genre_code, age_reference, esperance_annees
    FROM ext.INSEE_EsperanceVie
    WHERE age_reference IN (60, 65) AND annee BETWEEN 1994 AND 2023
    ORDER BY annee, genre_code, age_reference
""", conn)

for gc, color in [('F', 'salmon'), ('H', 'steelblue')]:
    sub60 = ev_full[(ev_full['genre_code'] == gc) & (ev_full['age_reference'] == 60)]
    sub65 = ev_full[(ev_full['genre_code'] == gc) & (ev_full['age_reference'] == 65)]
    if len(sub60):
        axes[1].plot(sub60['annee'], sub60['esperance_annees'], color=color, lw=2, ls='-', label=f'{gc} @60 ans')
    if len(sub65):
        axes[1].plot(sub65['annee'], sub65['esperance_annees'], color=color, lw=1.5, ls='--', label=f'{gc} @65 ans')

axes[1].set_title('Espérance de vie résiduelle à 60 et 65 ans', fontweight='bold')
axes[1].set_ylabel('Années restantes'); axes[1].legend(ncol=2, fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/09_duree_retraite.png', bbox_inches='tight')
plt.show()

## 4. Pression démographique — ratio dépendance vs effectifs CNAV

In [ ]:
pression = pd.read_sql("""
    SELECT p.annee,
           SUM(CASE WHEN p.age >= 65 AND p.genre_code IN ('H','F') THEN p.population END) * 1.0
           / NULLIF(SUM(CASE WHEN p.age BETWEEN 20 AND 64 AND p.genre_code IN ('H','F') THEN p.population END), 0)
               AS ratio_dep,
           d.retraites_total_cnav
    FROM ext.INSEE_PyramideAges p
    LEFT JOIN rpt.v_DashboardPrincipal d ON d.annee = p.annee
    WHERE p.annee BETWEEN 1991 AND 2040 AND p.genre_code IN ('H','F')
    GROUP BY p.annee, d.retraites_total_cnav
    ORDER BY p.annee
""", conn)

pression['ratio_dep_pct'] = pression['ratio_dep'] * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.fill_between(pression['annee'], pression['ratio_dep_pct'], alpha=0.2, color='crimson')
ax1.plot(pression['annee'], pression['ratio_dep_pct'], color='crimson', lw=2, label='Ratio dépendance (%)')

obs = pression[pression['retraites_total_cnav'].notna()]
ax2.plot(obs['annee'], obs['retraites_total_cnav'] / 1e6, color='steelblue', lw=2, ls='--', label='Effectifs CNAV (M)')

ax1.axvline(2024, color='gray', ls=':', lw=1)
ax1.set_title('Ratio de dépendance démographique vs effectifs CNAV', fontweight='bold')
ax1.set_ylabel('Ratio dep. (%)'); ax2.set_ylabel('Millions retraités CNAV')
ax1.grid(alpha=0.3)
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2)
plt.tight_layout()
plt.savefig('reports/09_pression_vs_cnav.png', bbox_inches='tight')
plt.show()

## 5. Résumé : sources et état du pipeline

In [ ]:
log = pd.read_sql("""
    SELECT source, indicateur, statut, nb_lignes, date_import
    FROM meta.SourcesExternesLog
    ORDER BY source, indicateur
""", conn)
print(log.to_string(index=False))

# Résumé des tables ext.*
print("\n=== Volumes ext.* ===")
vol = pd.read_sql("""
    SELECT s.name+'.'+t.name AS tbl, p.rows AS nb_lignes
    FROM sys.tables t JOIN sys.schemas s ON s.schema_id=t.schema_id
    JOIN sys.partitions p ON p.object_id=t.object_id AND p.index_id IN (0,1)
    WHERE s.name = 'ext'
    ORDER BY p.rows DESC
""", conn)
print(vol.to_string(index=False))

In [ ]:
conn.close()